# 📗 통계 기초 — 기술통계·확률·상관 분석

> 엔코아 AI캠퍼스 · 데이터 분석 & AI 머신러닝 캠프

그림으로 본 데이터를 이제 **숫자로 요약**합니다. 평균·표준편차로 분포를 정리하고, 두 변수의 관계를 **상관계수**로 재고, **확률**과 **확률분포**로 불확실성을 다루는 언어를 익힙니다.

## ⏪ 복습 — 지난 시간: EDA·시각화

지난 시간에는 데이터를 **그림으로** 살펴봤습니다.
- 히스토그램·KDE로 **분포**를, 산점도로 두 수치의 **관계**를, 히트맵으로 **집계표**를 눈으로 읽었습니다.
- 그림은 형태를 빠르게 보여 주지만, "평균이 얼마인지", "관계가 얼마나 강한지"를 **정확한 수치**로 말하지는 못합니다.

이번 시간에는 같은 데이터를 **숫자 한두 개로 요약**하는 법을 배웁니다. 그림이 보여 준 것을 수치로 확정하는 단계입니다.

**오늘의 목표**

- [ ] **대표값**(평균·중앙값·최빈값)으로 분포의 중심을 요약하고, 평균의 함정을 안다.
- [ ] **산포도**(분산·표준편차·IQR·변동계수)로 데이터가 얼마나 퍼졌는지 잰다.
- [ ] **왜도·첨도**로 분포의 치우침과 꼬리를 수치로 읽는다.
- [ ] **상관계수**(피어슨·스피어만)로 두 변수의 관계 방향과 세기를 잰다.
- [ ] **확률**과 **기대값**, 그리고 여러 **확률분포**(이항·포아송·정규 등)를 다룬다.
- [ ] **정규분포**의 68-95-99.7 법칙과 **Z-score**를 이해한다.
- [ ] 어떤 데이터에 **정규분포를 가정해도 되는지**(정규성)를 히스토그램·Q-Q Plot·왜도/첨도로 확인한다.

In [ ]:
# [제공 코드] 통계 분석에 쓸 라이브러리와 한글 폰트를 준비합니다.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import platform

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지
sns.set_theme(font=KOREAN_FONT, rc={'axes.unicode_minus': False})

---
# 통계적 사고 — 기술통계 vs 추론통계

통계는 크게 두 갈래입니다. **가진 데이터를 요약**하는 일과, 그 데이터로 **보지 못한 전체를 추측**하는 일입니다.

| 구분 | 기술통계 (descriptive) | 추론통계 (inferential) |
|---|---|---|
| 목적 | 가진 데이터를 **요약·묘사** | 표본으로 **모집단을 추측** |
| 질문 | "이 데이터는 어떻게 생겼나?" | "전체도 이럴까?" |
| 도구 | 평균·표준편차·상관·그래프 | 신뢰구간·가설검정 |
| 확실성 | 계산하면 **확정** | 항상 **불확실**(확률로 말함) |
| 다루는 시간 | 이번 시간 | 다음 시간 |

이번 시간은 **기술통계**(요약)와 그 바탕이 되는 **확률**을 배웁니다. 표본으로 전체를 추측하는 **추론통계**는 다음 시간의 몫입니다.

## 측정척도 — 숫자라고 다 같은 숫자가 아니다

값의 **성질**에 따라 할 수 있는 계산이 다릅니다. 척도를 구분하지 못하면 평균을 내면 안 되는 값의 평균을 내는 실수를 합니다.

| 척도 | 뜻 | 예시 | 가능한 계산 |
|---|---|---|---|
| 명목(nominal) | 이름·분류일 뿐 순서 없음 | 제조국(usa/japan/europe), 성별 | 개수·최빈값 (평균 불가) |
| 순서(ordinal) | 순서는 있으나 간격은 불명 | 만족도(상/중/하), 학점 | 중앙값·순위 |
| 등간(interval) | 간격은 같으나 **절대 0 없음** | 섭씨온도, 연도 | 덧셈·평균 (비율은 무의미) |
| 비율(ratio) | 간격도 같고 **0이 진짜 없음** | 연비·무게·나이 | 모든 계산 (배수 비교 가능) |

> 핵심 기준은 **"0이 진짜 없음을 뜻하는가"** 입니다. 무게 0kg은 진짜 없음(비율)이지만, 섭씨 0도는 없음이 아니라 기준점(등간)일 뿐입니다. 그래서 "20도는 10도의 2배로 덥다"는 틀린 말이지만 "20kg은 10kg의 2배 무겁다"는 맞습니다.

## 데이터 살펴보기 — 자동차 연비 데이터(mpg)

이번 시간 내내 쓸 데이터는 자동차 **연비(mpg)** 데이터입니다. 398대의 자동차에 대해 연비·실린더 수·배기량·마력·무게 등을 담았습니다.

| 열 이름 | 뜻 | 척도 |
|---|---|---|
| `mpg` | 연비(mile per gallon, 클수록 좋음) | 비율 |
| `cylinders` | 실린더 수(3·4·5·6·8) | 비율(사실상 순서) |
| `displacement` | 배기량 | 비율 |
| `horsepower` | 마력 (결측 6개) | 비율 |
| `weight` | 차량 무게 | 비율 |
| `acceleration` | 0→60mph 가속 시간 | 비율 |
| `model_year` | 출시 연도 | 등간 |
| `origin` | 제조국(usa/japan/europe) | 명목 |
| `name` | 차량 이름 | 명목 |

새 데이터를 만나면 분석에 앞서 **생김새부터** 확인합니다 — 앞부분(`head`), 구조·결측(`info`), 수치 요약(`describe`).

In [ ]:
mpg = pd.read_csv('data/mpg.csv')

print('데이터 크기:', mpg.shape)
print('\n[앞부분 5행]')
display(mpg.head())
print('\n[구조와 결측]')
mpg.info()
print('\n[수치형 요약]')
display(mpg.describe())
print('\n[범주형 요약]')
display(mpg.describe(exclude='number'))

---
# 1. 대표값 — 분포의 중심을 한 숫자로

## 왜 필요할까요?
398개의 연비 값을 하나하나 볼 수는 없습니다. **"대체로 얼마쯤인가"** 를 대표하는 한 숫자가 필요합니다. 이 중심 지표를 **대표값**이라고 합니다.

비유하면, 반 학생 40명의 키를 한마디로 말할 때 "대략 165cm" 라고 하는 것 — 그 한마디가 대표값입니다.

| 대표값 | 뜻 | 공식·문법 | 특징 |
|---|---|---|---|
| 산술평균 | 모두 더해 개수로 나눔 | `x.mean()` , x̄ = Σxᵢ / n | 가장 흔함. 이상치에 약함 |
| 중앙값 | 크기순 정렬 후 한가운데 값 | `x.median()` | 이상치에 **강건** |
| 최빈값 | 가장 자주 나온 값 | `x.mode()` | 범주형에도 사용 |
| 절사평균 | 상·하위 일부(%)를 버리고 평균 | `stats.trim_mean(x, 0.1)` | 이상치 완화 |
| 가중평균 | 중요도(가중치)를 반영한 평균 | `np.average(x, weights=w)` | 그룹 크기 반영 |

<img src="images/평균_중앙값_최빈값.png" width="780" style="max-width:100%">

> 위 그림처럼 분포가 **대칭**이면 평균≈중앙값≈최빈값이 겹치지만, 한쪽으로 **치우치면**(꼬리가 길면) 세 값이 벌어집니다. 오른쪽 꼬리가 길면 평균이 중앙값보다 **오른쪽**(큰 쪽)으로 끌려갑니다.

In [ ]:
# 연비(mpg)의 다섯 가지 대표값
x = mpg['mpg']
print('산술평균 :', round(x.mean(), 3))
print('중앙값   :', round(x.median(), 3))
print('최빈값   :', x.mode().tolist())
print('절사평균(상하 10% 제거):', round(stats.trim_mean(x, 0.1), 3))

# 가중평균 — 제조국별 평균 연비를 '대수(차량 수)'로 가중
origin_means = mpg.groupby('origin')['mpg'].mean()
origin_counts = mpg['origin'].value_counts()
weighted = np.average(origin_means, weights=origin_counts[origin_means.index])
print('\n제조국별 평균 연비:', origin_means.round(2).to_dict())
print('단순평균(그룹 평균의 평균):', round(origin_means.mean(), 3))
print('가중평균(대수 반영)      :', round(weighted, 3), '  ← 전체 평균과 일치')

## 평균의 함정 — 이상치가 평균을 끌어당긴다

평균은 **모든 값을 더하므로**, 아주 큰(또는 작은) 값 하나에 통째로 끌려갑니다. 예를 들어 직원 9명의 연봉이 3천만 원대인데 사장 한 명이 50억이면 "평균 연봉"은 5억이 넘어 버립니다 — 아무도 그만큼 받지 않는데도요.

이럴 때 **중앙값**은 한가운데 값이라 극단값 하나에 거의 흔들리지 않습니다(강건성). 아래에서 연비 데이터에 가상의 초고연비 차 한 대를 넣어 평균과 중앙값이 각각 얼마나 움직이는지 봅니다.

In [ ]:
# 연비 데이터에 이상치(가상의 초고연비 차 120) 하나를 추가
mpg_with_outlier = pd.concat([mpg['mpg'], pd.Series([120.0])], ignore_index=True)

print('원본        — 평균 {:.2f} / 중앙값 {:.2f}'.format(mpg['mpg'].mean(), mpg['mpg'].median()))
print('이상치 추가 — 평균 {:.2f} / 중앙값 {:.2f}'.format(mpg_with_outlier.mean(), mpg_with_outlier.median()))
print('\n값 하나 늘었을 뿐인데 평균은 크게 움직이고, 중앙값은 거의 그대로입니다.')

### 🖐️ 함께 따라하기 — 충전량의 대표값 구하기

데모는 **자동차 연비(mpg)** 로 봤습니다. 따라하기는 **다른 도메인 — 음료 제조 공장의 품질검사 기록**(`factory_quality.csv`)으로 같은 기술을 연습합니다. 생산 배치 480건마다 충전량·당도·라인속도·불량수를 적어 둔 표예요.

목표 충전량이 **500ml** 인 라인에서, 실제 충전량(`내용량_ml`)의 대표값을 구해 봅니다.

> ⚠️ 이 셀에서 만드는 `fac` 을 **이후 따라하기에서 계속 씁니다** — 건너뛰지 말고 꼭 실행하고 넘어가세요.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# ※ 데모는 '자동차 연비'였죠. 이번엔 다른 데이터(음료 공장 품질검사)로 연습합니다.
# 1) pd.read_csv 로 data/factory_quality.csv 를 읽어 fac 에 담는다
# 2) head() 로 앞부분을, info() 로 열·자료형·결측을 먼저 살펴본다
# 3) fill = fac['내용량_ml'] 을 만들고, 평균과 중앙값을 소수 둘째 자리로 출력한다
# 4) stats.trim_mean(fill, 0.1) 로 절사평균도 소수 둘째 자리로 출력한다
# 5) 목표가 500ml 인데 세 값이 왜 서로 다른지 생각해 본다

### ✅ 바로 확인 퀴즈

**1.** 소수의 아주 큰 값(이상치)이 섞여 있을 때, 중심을 나타내기에 더 안전한 대표값은 무엇인가요?

<details><summary>정답 보기</summary>

**중앙값**입니다. 평균은 모든 값을 더하므로 극단값 하나에 통째로 끌려가지만, 중앙값은 크기순 한가운데 값이라 이상치에 **강건**합니다.

</details>

**2.** 분포의 오른쪽 꼬리가 길 때(우로 치우침), 평균과 중앙값 중 어느 쪽이 더 큰가요?

<details><summary>정답 보기</summary>

**평균**이 더 큽니다. 오른쪽의 큰 값들이 평균을 오른쪽으로 끌어당기기 때문입니다(평균 > 중앙값 > 최빈값 순).

</details>

---
# 2. 산포도 — 얼마나 퍼져 있나

## 왜 필요할까요?
평균이 같아도 데이터가 **촘촘히 모였는지 넓게 흩어졌는지**는 전혀 다를 수 있습니다. 예를 들어 평균 수익률이 같은 두 투자 상품도, 하나는 값이 안정적이고 하나는 크게 출렁인다면 위험이 다릅니다. 이 **퍼짐의 정도**를 재는 것이 산포도입니다.

<img src="images/same_mean_other_std.png" width="780" style="max-width:100%">

| 지표 | 뜻 | 문법 |
|---|---|---|
| 범위(range) | 최댓값 − 최솟값 | `x.max() - x.min()` |
| IQR | Q3 − Q1 (가운데 50% 폭) | `stats.iqr(x)` |
| 분산(variance) | 편차 제곱의 평균 | `x.var(ddof=1)` |
| 표준편차(std) | 분산의 제곱근(원 단위로 복귀) | `x.std(ddof=1)` |
| 변동계수(CV) | 표준편차 ÷ 평균 (무단위 %) | `x.std()/x.mean()*100` |

### 베셀 보정 — 왜 n이 아니라 n−1로 나눌까?
표본분산은 편차 제곱합을 **n−1** 로 나눕니다(`ddof=1`). 표본으로 계산한 평균(x̄)을 기준으로 편차를 재면, 그 편차들은 이미 x̄에 맞춰져 있어 실제 모집단보다 **덜 퍼져** 보입니다(과소추정). n−1로 나눠 이 축소를 보정합니다 — 이것이 **베셀 보정**이고, n−1을 **자유도**라 부릅니다. 표본이 작을수록 이 보정의 효과가 큽니다. (참고로 모집단 전체면 `ddof=0`으로 n으로 나눕니다.)

### 변동계수(CV) — 단위가 다른 변수를 견줄 때
무게(kg)의 표준편차와 연비(mpg)의 표준편차를 그냥 비교하면 단위가 달라 무의미합니다. **CV = 표준편차 ÷ 평균**은 단위를 지운 **상대적 퍼짐**이라, 서로 다른 단위·스케일의 변수도 "어느 쪽이 더 들쭉날쭉한가"로 비교할 수 있습니다.

In [ ]:
# 연비의 산포도 지표들
x = mpg['mpg']
print('범위(range)      :', round(x.max() - x.min(), 2))
print('IQR              :', round(stats.iqr(x), 3))
print('표본분산(ddof=1) :', round(x.var(ddof=1), 3))
print('모분산(ddof=0)   :', round(x.var(ddof=0), 3), '  ← n으로 나눠 조금 작다')
print('표준편차(ddof=1) :', round(x.std(ddof=1), 3))

# 단위가 다른 세 변수를 변동계수(CV)로 비교
print('\n[변동계수 — 단위를 지운 상대적 퍼짐]')
for col in ['mpg', 'weight', 'acceleration']:
    values = mpg[col]
    cv = values.std(ddof=1) / values.mean() * 100
    print(f'  {col:13s}: 표준편차 {values.std(ddof=1):8.2f} → 변동계수 {cv:5.1f}%')

## 이상치 판별 — IQR 규칙과 Z-score

튀는 값(이상치)을 **눈이 아니라 규칙**으로 골라내는 두 가지 방법입니다.

- **IQR 1.5배 규칙**: Q1 − 1.5·IQR 보다 작거나 Q3 + 1.5·IQR 보다 크면 이상치 후보. 상자그림의 수염이 바로 이 경계입니다.
- **Z-score 규칙**: Z = (x − 평균) / 표준편차. |Z| 가 3(때로 2)을 넘으면 평균에서 지나치게 먼 값으로 봅니다.

<img src="images/box_plot.png" width="780" style="max-width:100%">

> 이상치라고 무조건 지우면 안 됩니다. 입력 오류인지, 드물지만 진짜인 값인지 **도메인 지식**으로 판단한 뒤 처리합니다.

In [ ]:
# 상자그림으로 이상치를 눈으로 본 뒤, 두 규칙으로 세어 본다
plt.figure(figsize=(9, 4))
ax = sns.boxplot(x=mpg['mpg'])
ax.set_title('연비 상자그림 — 수염 밖 점이 이상치 후보')
plt.show()

q1, q3 = mpg['mpg'].quantile(0.25), mpg['mpg'].quantile(0.75)
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
iqr_outliers = mpg[(mpg['mpg'] < lower) | (mpg['mpg'] > upper)]
print('IQR 경계: [{:.2f}, {:.2f}]'.format(lower, upper))
print('IQR 규칙 이상치 개수 :', len(iqr_outliers))

z_scores = (mpg['mpg'] - mpg['mpg'].mean()) / mpg['mpg'].std(ddof=1)
print('|Z| > 3 이상치 개수  :', int((z_scores.abs() > 3).sum()))
print('|Z| > 2 이상치 개수  :', int((z_scores.abs() > 2).sum()))

### 🖐️ 함께 따라하기 — 평균을 끌어내린 범인 찾기

앞 따라하기에서 충전량은 **평균만** 500ml 아래로 밀렸습니다. 그 **범인(이상치)을 IQR 규칙으로 직접 찾아**내고, 이어서 단위가 다른 두 열의 퍼짐을 **변동계수(CV)** 로 비교해 봅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) fill = fac['내용량_ml'] 에서 q1, q3 = fill.quantile(0.25), fill.quantile(0.75) 를 구한다
# 2) IQR = q3 - q1 로 이상치 하한(q1 - 1.5*IQR)·상한(q3 + 1.5*IQR)을 소수 첫째 자리로 출력한다
# 3) 하한보다 작거나 상한보다 큰 배치가 몇 건인지 세어 출력한다
#    (힌트: 두 조건을 | 로 묶고 .sum() — 조건마다 괄호를 씌우세요)
# 4) 내용량_ml 과 라인속도_bpm 의 변동계수(표준편차 / 평균 * 100)를 각각 출력해
#    단위가 다른 두 열의 퍼짐을 비교한다

### ✅ 바로 확인 퀴즈

**1.** 표본분산을 구할 때 편차 제곱합을 n이 아니라 **n−1** 로 나누는 이유는 무엇인가요?

<details><summary>정답 보기</summary>

표본평균을 기준으로 편차를 재면 퍼짐이 실제보다 **과소추정**되기 때문입니다. n−1(자유도)로 나눠 이를 보정하면 모분산에 대한 치우치지 않은(불편) 추정이 됩니다. 이를 **베셀 보정**이라 합니다.

</details>

**2.** 무게(kg)와 연비(mpg)처럼 **단위가 다른** 두 변수의 퍼짐을 비교하려면 어떤 지표가 알맞나요?

<details><summary>정답 보기</summary>

**변동계수(CV = 표준편차 ÷ 평균)** 입니다. 단위를 지운 상대적 퍼짐이라 단위·스케일이 다른 변수끼리도 비교할 수 있습니다.

</details>

---
# 3. 분포의 형태 — 왜도와 첨도

## 왜 필요할까요?
평균과 표준편차만으로는 분포의 **모양**(치우침, 꼬리)을 알 수 없습니다. 같은 평균·표준편차라도 좌우 대칭일 수도, 한쪽으로 쏠릴 수도 있습니다. 먼저 지난 시간에 배운 네 그림(히스토그램·KDE·상자·바이올린)으로 모양을 보고, 그 치우침과 꼬리를 **수치**로 확정합니다.

<img src="images/kde.png" width="760" style="max-width:100%">

<img src="images/violin_plot.png" width="760" style="max-width:100%">

In [ ]:
# 같은 연비 데이터를 네 가지 분포 그림으로 (히스토그램·KDE·상자·바이올린)
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
sns.histplot(data=mpg, x='mpg', bins=20, kde=True, ax=axes[0, 0])
axes[0, 0].set_title('히스토그램 (+ KDE)')
sns.kdeplot(data=mpg, x='mpg', fill=True, ax=axes[0, 1])
axes[0, 1].set_title('밀도곡선(KDE)')
sns.boxplot(x=mpg['mpg'], ax=axes[1, 0])
axes[1, 0].set_title('상자그림')
sns.violinplot(x=mpg['mpg'], ax=axes[1, 1])
axes[1, 1].set_title('바이올린')
fig.tight_layout()
plt.show()

## 왜도(skewness)와 첨도(kurtosis)

- **왜도**: 분포가 **좌우로 얼마나 치우쳤나**. `stats.skew(x)`
  - 왜도 ≈ 0 → 대칭 / 왜도 > 0 → **오른쪽 꼬리**(연봉·집값처럼 큰 값이 드물게) / 왜도 < 0 → 왼쪽 꼬리
  - 공식(개념): 표준화한 편차의 **세제곱** 평균 — 방향(부호)이 살아남습니다.
- **첨도**: 분포의 **꼬리가 얼마나 두꺼운가**(극단값이 얼마나 자주). `stats.kurtosis(x)`
  - scipy `stats.kurtosis` 는 **초과첨도**를 줍니다(정규분포 기준 0). > 0 → 정규보다 뾰족하고 꼬리 두꺼움 / < 0 → 완만.
  - 공식(개념): 표준화한 편차의 **네제곱** 평균 − 3.
  - pandas의 `.skew()`·`.kurt()`도 같은 개념(초과첨도)이지만 **표본 보정 방식이 달라 값이 살짝 다릅니다** — 이 교안은 scipy `stats.skew`·`stats.kurtosis` 로 통일합니다.

<img src="images/왜도.png" width="820" style="max-width:100%">

<img src="images/첨도.png" width="480" style="max-width:100%">

> **꼬리 리스크**: 첨도가 큰(꼬리가 두꺼운) 분포는 "거의 안 일어나지만 한 번 터지면 큰" 사건이 정규분포 예상보다 자주 옵니다 — 2008년 금융위기처럼요. 그래서 꼬리 두께를 살피는 일이 중요합니다.

In [ ]:
# 세 변수의 왜도·첨도를 수치로 (scipy 기준 — 과제 채점과 동일한 방식)
for col in ['mpg', 'weight', 'acceleration']:
    x = mpg[col].dropna()
    print(f'{col:13s}: 왜도 {stats.skew(x):+.3f}, 첨도 {stats.kurtosis(x):+.3f}')

# 연비 분포의 치우침을 밀도곡선으로 확인 (왜도 > 0 → 오른쪽 꼬리)
skew_value = stats.skew(mpg['mpg'])
plt.figure(figsize=(8, 4))
ax = sns.kdeplot(data=mpg, x='mpg', fill=True)
ax.axvline(mpg['mpg'].mean(), color='red', linestyle='--', label='평균')
ax.axvline(mpg['mpg'].median(), color='green', linestyle='--', label='중앙값')
ax.set_title(f'연비 분포 — 왜도 {skew_value:.2f} (오른쪽 꼬리)')
ax.legend()
plt.show()

### 🖐️ 함께 따라하기 — 설비 알람 횟수의 치우침 재기

한 배치를 돌리는 동안 울린 **설비 알람 횟수(`설비알람수`)** 의 분포를 그리고 치우침을 수치로 재 봅니다. 대부분의 배치는 알람이 거의 없지만 가끔 많이 울리는 배치가 있죠 — 그 모양이 왜도·첨도로 어떻게 나타날까요?

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) plt.figure(figsize=(7,4)) 로 새 도화지를 연다
# 2) ax = sns.histplot(data=fac, x='설비알람수', discrete=True) 로 분포를 그린다
#    (알람 횟수는 정수라 discrete=True 가 자연스럽습니다)
# 3) ax.set_title('배치별 설비 알람 횟수 분포') 로 제목을 달고 plt.show()
# 4) stats.skew 와 stats.kurtosis 로 왜도·첨도를 소수 셋째 자리로 출력한다
# 5) 왜도의 부호를 보고 꼬리가 어느 쪽으로 뻗었는지 말해 본다

### ✅ 바로 확인 퀴즈

**1.** 어떤 분포의 왜도가 **양수(+)** 로 나왔습니다. 꼬리는 어느 쪽으로 길고, 평균과 중앙값 중 무엇이 더 큰가요?

<details><summary>정답 보기</summary>

꼬리는 **오른쪽**으로 깁니다(큰 값이 드물게 존재). 오른쪽의 큰 값들이 평균을 끌어올리므로 **평균 > 중앙값** 입니다.

</details>

**2.** scipy `stats.kurtosis()`(초과첨도)가 **양수**면 정규분포와 비교해 꼬리가 어떤가요?

<details><summary>정답 보기</summary>

정규분포(기준 0)보다 **꼬리가 두껍고** 봉우리가 더 뾰족합니다. 즉 극단값이 정규분포 예상보다 자주 나타납니다(꼬리 리스크).

</details>

---
# 4. 상관 분석 — 두 변수는 함께 움직이나

## 왜 필요할까요?
지난 시간엔 산점도로 두 변수의 관계를 **눈으로** 봤습니다. 이제 그 관계의 **방향과 세기를 하나의 숫자**(상관계수)로 잽니다. 예를 들어 "차가 무거울수록 연비가 나빠진다"는 직관을 수치로 확정할 수 있습니다.

먼저 무게와 연비의 산점도를 다시 그려, 점들이 **오른쪽 아래로** 늘어서는지 눈으로 확인합니다.

In [ ]:
# 무게 vs 연비 — 점이 오른쪽 아래로 늘어서면 '무거울수록 연비 낮음'(음의 관계)
plt.figure(figsize=(7, 5))
ax = sns.scatterplot(data=mpg, x='weight', y='mpg')
ax.set_title('차량 무게 vs 연비')
plt.show()

## 피어슨 상관계수

**피어슨 상관계수 r** 은 두 변수의 **직선 관계**의 방향과 세기를 −1 ~ +1 사이 한 숫자로 나타냅니다.

- r > 0 → 한쪽이 커지면 다른 쪽도 커짐(양의 관계) / r < 0 → 반대(음의 관계) / r ≈ 0 → 직선 관계 약함
- `stats.pearsonr(x, y)` 는 **두 값**을 돌려주므로 `[0]` 으로 상관계수만 받습니다. 여러 변수를 한 번에 보려면 `df.corr(numeric_only=True)`.

| \|r\| 범위 | 해석 |
|---|---|
| 0.0 ~ 0.3 | 약한 관계 |
| 0.3 ~ 0.7 | 중간 관계 |
| 0.7 ~ 1.0 | 강한 관계 |

<img src="images/상관계수_비교.png" width="780" style="max-width:100%">

In [ ]:
# 피어슨 상관계수 — 무게와 연비
r = stats.pearsonr(mpg['weight'], mpg['mpg'])[0]
print('무게-연비 피어슨 r =', round(r, 3), '  (강한 음의 관계)')

# 수치형 변수 전체의 상관행렬
print('\n[연비(mpg)와 각 변수의 상관계수]')
corr_matrix = mpg.corr(numeric_only=True)
display(corr_matrix['mpg'].round(3).to_frame('연비와의 상관'))

## 스피어만 상관계수 — 순위로 재는 관계

피어슨은 **직선** 관계를 재고 이상치에 민감합니다. **스피어만 상관계수 rho** 는 값 대신 **순위(등수)** 로 계산해, 곡선이라도 "한쪽이 커질수록 다른 쪽도 커지는"(단조) 관계를 잡아내고 **이상치에 강건**합니다. `stats.spearmanr(x, y)`.

In [ ]:
# 스피어만(순위 기반) 상관 — 이상치·비직선 관계에 강건
rho, _ = stats.spearmanr(mpg['weight'], mpg['mpg'])
r_pearson, _ = stats.pearsonr(mpg['weight'], mpg['mpg'])
print('피어슨  r   =', round(r_pearson, 3))
print('스피어만 rho =', round(rho, 3), '  (순위로 재면 관계가 더 뚜렷)')

## 상관행렬 히트맵 — 모든 쌍을 한눈에

변수가 많으면 상관행렬을 **히트맵**으로 색칠해 한눈에 봅니다. `annot=True` 로 숫자를, `cmap='coolwarm'` 으로 양(빨강)·음(파랑)을 색으로 구분합니다.

In [ ]:
# 상관행렬 히트맵 — 빨강=양의 관계, 파랑=음의 관계
plt.figure(figsize=(8, 6))
corr_matrix = mpg.corr(numeric_only=True)
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title('수치형 변수 상관행렬')
plt.show()

## 상관계수의 한계 — 숫자만 믿지 말고 산점도를

상관계수는 편리하지만 관계를 **한 숫자로 요약**하는 만큼 놓치는 것이 많습니다. r 을 볼 때는 **반드시 산점도를 함께** 봐야 합니다.

- **① 선형(단조) 관계만 잡는다**: 피어슨 r 은 **직선** 관계의 강도만 잽니다. U자·곡선처럼 **비선형** 관계는 관계가 뚜렷해도 r ≈ 0 이 나올 수 있습니다(스피어만도 **단조**가 아니면 못 잡습니다).
- **② 이상치에 민감하다**: 점 한두 개가 r 을 크게 흔듭니다. 약한 음의 상관이 이상치 하나로 양의 상관처럼 보이기도 합니다.
- **③ 같은 r, 다른 그림**: 상관계수 값이 같아도 산점도 모양은 전혀 다를 수 있습니다(같은 r=0.82 이라도 곧은 직선일 수도, 곡선일 수도, 이상치 하나 때문일 수도) — 숫자만 믿지 말라는 유명한 교훈입니다(아래 그림).
- **④ 범위가 좁으면 과소평가**: 데이터 범위를 일부만 잘라 보면 실제보다 상관이 약해 보일 수 있습니다.

아래 **앤스컴 콰르텟(Anscombe's quartet)** 이 ③의 대표 예입니다 — 네 데이터가 **평균·상관계수(r=0.82)·회귀선까지 거의 같은데도** 산점도 모양은 전혀 다릅니다. ①만 진짜 직선 관계이고, ②는 곡선, ③·④는 이상치 하나가 만든 착시입니다.

<img src="images/anscombe_quartet.png" width="860" style="max-width:100%">

> 결론: r 은 관계를 **한 숫자로 요약**할 뿐입니다. **비선형·이상치·범위**를 놓치지 않으려면 항상 산점도를 먼저 그려 눈으로 확인하세요. (원인·결과 문제는 바로 아래에서 따로 다룹니다.)

In [ ]:
# 상관계수의 함정 두 가지를 눈으로 확인
rng = np.random.default_rng(0)

# ① 비선형(U자형): 관계는 뚜렷한데 피어슨 r ≈ 0
x_nl = np.linspace(-3, 3, 200)
y_nl = x_nl**2 + rng.normal(0, 0.5, 200)
r_pearson = stats.pearsonr(x_nl, y_nl)[0]
r_spearman = stats.spearmanr(x_nl, y_nl)[0]
print(f'① 비선형: 피어슨 r = {r_pearson:.3f}, 스피어만 r = {r_spearman:.3f}  → 둘 다 0 근처지만 관계는 뚜렷!')

plt.figure(figsize=(8, 4))
plt.scatter(x_nl, y_nl, alpha=0.5)
plt.title('① 비선형(U자형) — 뚜렷한 관계지만 r ≈ 0')
plt.xlabel('x'); plt.ylabel('y')
plt.show()

In [ ]:
# ② 이상치 하나가 상관을 뒤집는다
x_out = rng.normal(0, 1, 30)
y_out = rng.normal(0, 1, 30)
r_before = stats.pearsonr(x_out, y_out)[0]
x_out2 = np.append(x_out, 8)
y_out2 = np.append(y_out, 8)
r_after = stats.pearsonr(x_out2, y_out2)[0]
print(f'② 이상치 전 r = {r_before:.3f} → 이상치 (8, 8) 하나 추가 후 r = {r_after:.3f}')

plt.figure(figsize=(8, 4))
plt.scatter(x_out2, y_out2, alpha=0.6)
plt.scatter([8], [8], color='red', s=80, label='이상치 (8, 8)')
plt.title(f'② 이상치 하나로 r 이 {r_before:.2f} → {r_after:.2f} 로 급변')
plt.xlabel('x'); plt.ylabel('y')
plt.legend()
plt.show()

## 상관 ≠ 인과 — 가장 중요한 주의

두 변수가 함께 움직인다고 해서 **한쪽이 다른 쪽의 원인**인 것은 아닙니다.

- 예: 여름이면 **아이스크림 판매**와 **물놀이 사고**가 함께 늘지만, 아이스크림이 사고를 일으키는 것은 아닙니다. 숨은 **제3의 변수(기온)** 가 둘 다를 끌어올린 것뿐입니다.
- 연비 데이터에서도 무게와 연비의 강한 음의 상관 뒤에는 배기량·실린더 수 같은 다른 변수가 함께 얽혀 있습니다.

> 상관은 **관계가 있다**까지만 말합니다. **원인**을 주장하려면 실험 설계나 추가 분석이 필요합니다.

### 🖐️ 함께 따라하기 — 라인속도와 충전량의 관계

라인을 빠르게 돌리면 충전이 덜 될까요? **라인속도(`라인속도_bpm`)** 와 **충전량(`내용량_ml`)** 의 관계를 **피어슨과 스피어만 두 가지로** 재 보고, 산점도로 눈으로도 확인합니다.

> 두 계수가 꽤 다르게 나올 겁니다. **왜 다른지**가 이 따라하기의 핵심이에요 — 방금 배운 '상관계수의 한계'를 실제 데이터에서 만나는 순간입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) stats.pearsonr(fac['라인속도_bpm'], fac['내용량_ml']) 의 첫 값을 r 에 담아 소수 셋째 자리로 출력한다
# 2) stats.spearmanr 로 스피어만 계수도 같은 방식으로 구해 출력한다
# 3) plt.figure(figsize=(7,4)) 후 sns.scatterplot(data=fac, x='라인속도_bpm', y='내용량_ml') 로 산점도를 그린다
# 4) 제목 '라인속도 vs 충전량' 을 달고 plt.show()
# 5) 산점도 아래쪽에 뚝 떨어진 점들을 보고, 두 계수가 왜 다른지 생각해 본다

### ✅ 바로 확인 퀴즈

**1.** 무게와 연비의 피어슨 상관계수가 약 **−0.83** 으로 나왔습니다. 어떻게 해석하나요?

<details><summary>정답 보기</summary>

부호가 음수이므로 **무거울수록 연비가 낮은** 관계이고, |r|이 0.7을 넘으니 **강한** 관계입니다. 즉 무게와 연비는 강한 음의 상관을 가집니다.

</details>

**2.** 아이스크림 판매량과 물놀이 사고 건수의 상관이 높게 나왔습니다. "아이스크림이 사고의 원인"이라고 말할 수 있나요?

<details><summary>정답 보기</summary>

아니요. **상관은 인과가 아닙니다.** 여기서는 숨은 제3의 변수인 **기온**이 둘 다를 끌어올린 것입니다. 상관은 관계의 존재만 말할 뿐 원인을 증명하지 못합니다.

</details>

**3.** 어떤 두 변수의 피어슨 상관계수가 **0 에 가깝게** 나왔습니다. "두 변수는 아무 관계가 없다"고 결론지어도 될까요?

<details><summary>정답 보기</summary>

아니요. 피어슨 r 은 **직선(선형) 관계**만 잽니다. U자·곡선 같은 **비선형** 관계는 뚜렷해도 r ≈ 0 이 나올 수 있습니다. r 이 0 이라도 **산점도를 반드시 확인**해야 하며, 이상치나 좁은 범위도 r 을 왜곡할 수 있습니다.

</details>

---
# 5. 확률 기초

## 왜 필요할까요?
추론통계는 "이 정도 차이가 우연일 확률"처럼 **확률의 언어**로 판단합니다. 그 바탕을 먼저 다집니다.

**빈도적 확률**은 "같은 시행을 무한히 반복하면 어떤 사건이 나오는 **상대빈도**"로 확률을 정의합니다. 동전을 던질수록 앞면 비율이 0.5에 가까워지는 것 — 이 수렴을 **대수의 법칙(큰 수의 법칙)** 이라 합니다.

<img src="images/빈도적_확률_수렴.png" width="780" style="max-width:100%">

In [ ]:
# 대수의 법칙 — 동전을 많이 던질수록 앞면 비율이 0.5로 수렴
rng = np.random.default_rng(42)
tosses = rng.choice([0, 1], size=10000)          # 0=뒷면, 1=앞면
running_ratio = np.cumsum(tosses) / np.arange(1, 10001)

plt.figure(figsize=(8, 4))
plt.plot(running_ratio)
plt.axhline(0.5, color='red', linestyle='--', label='이론 확률 0.5')
plt.xscale('log')
plt.title('동전 앞면 비율 — 시행이 늘수록 0.5로 수렴')
plt.xlabel('시행 횟수(로그 축)')
plt.ylabel('앞면 비율')
plt.legend()
plt.show()

for n in [10, 100, 1000, 10000]:
    print(f'{n:>5}회 던짐 → 앞면 비율 {tosses[:n].mean():.3f}')

## 확률의 기본 법칙

주사위 한 개(1~6)를 예로 봅니다.

- **여사건**: P(Aᶜ) = 1 − P(A). "짝수가 아닐 확률" = 1 − P(짝수).
- **덧셈법칙**: P(A ∪ B) = P(A) + P(B) − P(A ∩ B). 겹치는 부분을 한 번 빼 줍니다.
- **곱셈법칙**: P(A ∩ B) = P(A) · P(B\|A).
- **조건부확률**: P(A\|B) = P(A ∩ B) / P(B). "B가 일어났다는 조건에서 A".
- **독립**: 한 사건이 다른 사건에 영향을 주지 않음 → P(A ∩ B) = P(A)·P(B). (동전 두 번 던지기)
- **배반(상호배타)**: 두 사건이 동시에 못 일어남 → P(A ∩ B) = 0. (한 번 던져 짝수이면서 홀수)

> **독립과 배반은 다릅니다.** 배반이면 겹침이 0이라 오히려 서로 강하게 얽혀 있습니다(하나가 나오면 다른 하나는 불가능).

In [ ]:
# 주사위 한 개로 확률 법칙 확인
p_even = 3 / 6            # 짝수 {2,4,6}
p_ge5 = 2 / 6            # 5 이상 {5,6}
p_even_and_ge5 = 1 / 6   # 교집합 {6}

print('P(짝수)              =', round(p_even, 3))
print('P(짝수 또는 5이상)   =', round(p_even + p_ge5 - p_even_and_ge5, 3), '  (덧셈법칙)')
print('P(짝수 | 5이상)      =', round(p_even_and_ge5 / p_ge5, 3), '  (조건부확률)')
print('P(두 번 던져 모두 6) =', round((1 / 6) * (1 / 6), 4), '  (독립: 곱셈)')

## 기대값과 분산

**기대값 E[X]** 은 "확률로 가중한 평균" — 같은 시행을 무한히 반복하면 평균적으로 얻는 값입니다.

- 이산: E[X] = Σ x · P(X = x), 분산: Var(X) = Σ (x − E[X])² · P(X = x)
- 성질: E[aX + b] = a·E[X] + b, Var(aX + b) = a²·Var(X)

<img src="images/기대값_분산.png" width="780" style="max-width:100%">

> 복권·보험이 대표 예입니다. 기대값이 음수면(살 때마다 평균 손해) 장기적으로 손실입니다.

In [ ]:
# 주사위 눈의 기대값과 분산
outcomes = np.array([1, 2, 3, 4, 5, 6])
probs = np.ones(6) / 6
expected = np.sum(outcomes * probs)
variance = np.sum((outcomes - expected) ** 2 * probs)
print('기대값 E[X] =', expected)
print('분산 Var(X) =', round(variance, 3))
print('표준편차     =', round(np.sqrt(variance), 3))

### 🖐️ 함께 따라하기 — 불량 배치 처리비용의 기대값

데모에서는 주사위 눈의 기대값을 구했죠. 이번엔 **공장의 실제 비용**으로 기대값을 계산합니다.

검사 100개당 불량수에 따라 배치 처리 방침이 다릅니다.

| 불량수 | 조치 | 비용 |
|---|---|---|
| 2개 이하 | 정상 출하 | 0원 |
| 3~5개 | 전수 재검사 | 500,000원 |
| 6개 이상 | 배치 폐기 | 3,000,000원 |

각 경우의 **확률을 데이터에서 직접 구해**, 배치 하나당 기대 처리비용을 계산해 봅시다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) d = fac['불량수'] 를 만든다
# 2) 세 경우의 확률을 데이터에서 구한다 — 조건에 맞는 비율은 (조건).mean() 으로 바로 나온다
#    p_ok = (d <= 2).mean() / p_re = ((d >= 3) & (d <= 5)).mean() / p_out = (d >= 6).mean()
# 3) 세 확률의 합이 1인지 확인해 출력한다
# 4) 기대비용 = 0*p_ok + 500000*p_re + 3000000*p_out 을 계산해 출력한다
# 5) 배치 480건을 돌리면 총 얼마가 예상되는지도 곱해서 확인한다

### ✅ 바로 확인 퀴즈

**1.** 주사위를 두 번 던져 **두 번 다 6** 이 나올 확률은? (두 번은 서로 독립)

<details><summary>정답 보기</summary>

독립이므로 곱셈법칙으로 (1/6) × (1/6) = **1/36 ≈ 0.028** 입니다.

</details>

**2.** "동시에 일어날 수 없다"는 **배반**과 "서로 영향이 없다"는 **독립**은 같은 뜻인가요?

<details><summary>정답 보기</summary>

아니요, 다릅니다. **배반**은 교집합 확률이 0(동시에 못 일어남), **독립**은 P(A∩B)=P(A)·P(B)(서로 영향 없음)입니다. 배반 사건은 오히려 강하게 얽혀 있어(하나가 나오면 다른 하나는 불가능) 독립이 아닙니다.

</details>

---
# 6. 확률분포 — PMF·PDF·CDF

## 왜 필요할까요?
확률변수가 가질 수 있는 값과 그 확률을 한꺼번에 그린 것이 **확률분포**입니다. 현실의 현상을 알려진 분포로 모델링하면 "이런 값이 나올 확률"을 계산할 수 있습니다.

- **이산형**은 값이 뚝뚝 떨어짐(주사위·불량 개수) → **PMF**(Probability Mass Function, 확률질량함수): 각 값의 확률.
- **연속형**은 값이 이어짐(키·시간) → **PDF**(Probability Density Function, 확률밀도함수): 넓이가 확률.
- **CDF**(Cumulative Distribution Function, 누적분포함수) F(x) = P(X ≤ x): x 이하일 확률. 이산은 계단, 연속은 부드러운 곡선.

<img src="images/이산_연속_비교.png" width="900" style="max-width:100%"> \

<img src="images/누적분포함수_CDF.png" width="900" style="max-width:100%">

scipy는 분포 객체에 공통 메서드를 줍니다: `.pmf/.pdf`(확률), `.cdf`(이하 누적), `.sf`(Survival Function, 초과 =1−cdf), `.ppf`(Percent Point Function, 분위수), `.mean/.var/.std` (기댓값, 분산, 표준편차).

<img src="images/분포_선택_가이드.png" width="1600" style="max-width:100%">

In [ ]:
# 주사위: PMF(각 눈의 확률)와 CDF(누적확률 = PMF의 누적합)
x_dice = np.arange(1, 7)
pmf_dice = np.ones(6) / 6
cdf_dice = np.cumsum(pmf_dice)

# PMF — 각 눈이 나올 확률
plt.figure(figsize=(9, 4))
plt.bar(x_dice, pmf_dice)
plt.ylim(0, 0.3)
plt.title('PMF — 각 눈이 나올 확률')
plt.xlabel('눈')
plt.ylabel('확률')
plt.show()

# CDF — 누적확률 P(X ≤ x)
plt.figure(figsize=(9, 4))
plt.step(x_dice, cdf_dice, where='mid')
plt.title('CDF — 누적확률 P(X ≤ x)')
plt.xlabel('눈')
plt.ylabel('누적확률')
plt.show()
print('P(X ≤ 3) =', round(cdf_dice[2], 3))

## 확률 계산 정리 — cdf · sf · ppf (이산형 vs 연속형)

"몇 이하일 확률", "몇 초과일 확률", "상위 몇 %의 경계" 같은 물음을 scipy 메서드로 바꾸는 표입니다. **가장 중요한 차이**: 연속형은 P(X=x)=0 이라 **'이하(≤)'와 '미만(<)'이 같지만**, 이산형은 P(X=k)>0 이라 **다릅니다** — 경계값을 포함하는지 반드시 따져야 합니다.

**공통 메서드**
- `cdf(x)` = P(X ≤ x) — x 이하일 누적확률
- `sf(x)` = P(X > x) = 1 − cdf(x) — x 초과일 확률
- `ppf(q)` = 누적확률이 q 가 되는 값 — 분위수(cdf 의 반대)

**연속형 (정규·지수·균등) — ≤ 와 < 가 같다**

| 구하려는 확률 | scipy |
|---|---|
| P(X ≤ x) = P(X < x) | `dist.cdf(x)` |
| P(X ≥ x) = P(X > x) | `dist.sf(x)`  (= 1 − cdf(x)) |
| P(a ≤ X ≤ b) | `dist.cdf(b) - dist.cdf(a)` |
| 하위 p 비율의 경계값 | `dist.ppf(p)` |
| 상위 p 비율의 경계값 (상위 5% → p=0.05) | `dist.ppf(1 - p)` |

**이산형 (이항·포아송, k 는 정수) — ≤ 와 < 가 다르다 (경계 포함 주의)**

| 구하려는 확률 | scipy | 왜 |
|---|---|---|
| P(X = k) | `dist.pmf(k)` | 딱 k |
| P(X ≤ k) | `dist.cdf(k)` | k 포함 |
| P(X < k) | `dist.cdf(k - 1)` | k 제외 = k−1 이하 |
| P(X ≥ k) | `dist.sf(k - 1)` | k 포함 = 1 − cdf(k−1) |
| P(X > k) | `dist.sf(k)` | k 제외 |
| P(a ≤ X ≤ b) | `dist.cdf(b) - dist.cdf(a - 1)` | 양끝 포함 |

> **자주 하는 실수**: 이산형에서 "k 이상"(P(X ≥ k))을 `sf(k)` 로 쓰면 **k 를 빼먹습니다**. `sf(k)` 는 P(X > k)(초과)이므로, 이상은 `sf(k-1)` 이 맞습니다. 연속형에서는 이 구분이 없습니다.

In [ ]:
# 같은 값 5를 두고 '이하/미만/이상/초과'가 이산·연속에서 어떻게 갈리는지 확인
binom_ex = stats.binom(n=20, p=0.3)      # 이산형 (값이 정수)
norm_ex = stats.norm(loc=5, scale=2)     # 연속형

print('[이산형 B(20, 0.3)]  — 이하(≤)와 미만(<)이 다르다')
print('  P(X ≤ 5) = cdf(5) =', round(binom_ex.cdf(5), 4))
print('  P(X < 5) = cdf(4) =', round(binom_ex.cdf(4), 4), ' (5 제외)')
print('  P(X ≥ 5) = sf(4)  =', round(binom_ex.sf(4), 4), ' (5 포함)')
print('  P(X > 5) = sf(5)  =', round(binom_ex.sf(5), 4), ' (5 제외)')
print('  P(X = 5) = pmf(5) =', round(binom_ex.pmf(5), 4))
print()
print('[연속형 N(5, 2²)]  — 이하(≤)와 미만(<)이 같다')
print('  P(X ≤ 5) = P(X < 5) = cdf(5) =', round(norm_ex.cdf(5), 4))
print('  P(X ≥ 5) = P(X > 5) = sf(5)  =', round(norm_ex.sf(5), 4))
print('  하위 90% 경계 = ppf(0.90)     =', round(norm_ex.ppf(0.90), 4))

## 이산형 분포 — 값이 뚝뚝 떨어진다

이산형 확률변수는 셀 수 있는 값(0, 1, 2, …)을 가집니다. 대표적인 세 분포 **베르누이·이항·포아송**을 **하나씩** 보겠습니다.

### 베르누이 분포 — 성공/실패 한 번

동전 한 번, 합격/불합격처럼 **딱 한 번의 시행에서 성공(1)·실패(0)** 만 나오는 분포입니다. 성공확률 p 하나로 정해집니다.

- 기대값 = p, 분산 = p(1−p), scipy `stats.bernoulli(p)`

<img src="images/베르누이_예시.png" width="780" style="max-width:100%">

In [ ]:
# 베르누이: 합격확률 0.3인 시험을 한 번 (성공=1, 실패=0)
bern_dist = stats.bernoulli(p=0.3)
print('P(합격=1) =', round(bern_dist.pmf(1), 3), ' / P(불합격=0) =', round(bern_dist.pmf(0), 3))
print('기대값 =', bern_dist.mean(), ' / 분산 =', round(bern_dist.var(), 3))

plt.figure(figsize=(9, 4))
plt.bar([0, 1], bern_dist.pmf([0, 1]))
plt.xticks([0, 1], ['실패(0)', '성공(1)'])
plt.title('베르누이 분포 (p=0.3)')
plt.ylabel('확률')
plt.show()

### 이항 분포 — n번 중 성공 수

성공확률 p인 베르누이 시행을 **독립으로 n번** 반복했을 때 **성공 횟수**의 분포입니다. 불량률 p로 n개를 검사할 때의 불량 개수 등.

- 기대값 = np, 분산 = np(1−p), scipy `stats.binom(n, p)`

<img src="images/이항분포_예시.png" width="780" style="max-width:100%">

In [ ]:
# 이항: 불량률 5%로 20개를 뽑을 때 불량 개수
binom_dist = stats.binom(n=20, p=0.05)
print('기대 불량 수 =', binom_dist.mean(), ' / P(불량 0개) =', round(binom_dist.pmf(0), 4))
print('P(불량 2개 이하) =', round(binom_dist.cdf(2), 4))
print('P(불량 3개 이상) =', round(binom_dist.sf(2), 4), '  (sf = 1 - cdf)')

k = np.arange(0, 11)
plt.figure(figsize=(9, 4))
plt.bar(k, binom_dist.pmf(k))
plt.title('이항분포 B(20, 0.05)')
plt.xlabel('불량 개수')
plt.ylabel('확률')
plt.show()

### 포아송 분포 — 드문 사건의 횟수

단위 시간·공간에서 **평균 λ번 일어나는 드문 사건**의 횟수 분포입니다(콜센터 문의 수, 웹서버 접속 수). **기대값과 분산이 모두 λ로 같은** 것이 특징입니다.

- 기대값 = λ, 분산 = λ, scipy `stats.poisson(mu=λ)`

<img src="images/포아송_예시.png" width="780" style="max-width:100%">

In [ ]:
# 포아송: 콜센터에 시간당 평균 3콜
pois_dist = stats.poisson(mu=3)
print('P(정확히 5콜) =', round(pois_dist.pmf(5), 4), ' / P(2콜 이하) =', round(pois_dist.cdf(2), 4))
print('기대값 =', pois_dist.mean(), ' / 분산 =', pois_dist.var(), '  (둘이 같다)')

k = np.arange(0, 11)
plt.figure(figsize=(9, 4))
plt.bar(k, pois_dist.pmf(k))
plt.title('포아송분포 λ=3')
plt.xlabel('사건 횟수')
plt.ylabel('확률')
plt.show()

## 연속형 분포 — 값이 이어진다

연속형 확률변수는 이어진 값(키, 시간)을 가지며, 확률은 **넓이(PDF)** 로 읽습니다. 대표적인 세 분포 **균등·지수·정규**를 **하나씩** 보겠습니다.

### 균등 분포 — 구간 안에서 고르게

구간 [a, b] 안의 모든 값이 **똑같이 나올 법한** 분포입니다(버스가 0~10분 사이 아무 때나 도착). 밀도가 평평합니다.

- 기대값 = (a+b)/2, scipy `stats.uniform(loc=a, scale=b−a)`

<img src="images/균등분포_예시.png" width="780" style="max-width:100%">

In [ ]:
# 균등: 버스가 0~10분 사이 균등하게 도착
unif_dist = stats.uniform(loc=0, scale=10)
print('평균 대기 {:.1f}분, P(3분 이내 도착) {:.2f}'.format(unif_dist.mean(), unif_dist.cdf(3)))

grid = np.linspace(-1, 11, 200)
plt.figure(figsize=(9, 4))
plt.plot(grid, unif_dist.pdf(grid))
plt.title('균등분포 U(0, 10)')
plt.xlabel('대기 시간(분)')
plt.ylabel('확률밀도')
plt.show()

### 지수 분포 — 사건 사이의 대기 시간

포아송이 "횟수"라면 지수는 그 사건들 **사이의 간격(대기 시간)** 분포입니다(고장까지의 수명). **무기억성**(이미 800시간 버텼어도 남은 수명 분포는 처음과 같음)을 가집니다. 포아송(횟수)과 지수(간격)는 동전의 양면입니다.

- 기대값 = 1/λ, scipy `stats.expon(scale=1/λ)`

> **주의**: scipy는 `scale` 에 **평균 대기시간(1/λ)** 을 넣습니다. "평균 수명 1000시간"이면 `stats.expon(scale=1000)`.

<img src="images/지수분포_예시.png" width="780" style="max-width:100%">

In [ ]:
# 지수: 평균 수명 1000시간인 부품
expon_dist = stats.expon(scale=1000)
print('P(500h 이내 고장) {:.4f}, 중앙 수명 {:.1f}h'.format(expon_dist.cdf(500), expon_dist.median()))

grid = np.linspace(0, 4000, 200)
plt.figure(figsize=(9, 4))
plt.plot(grid, expon_dist.pdf(grid))
plt.title('지수분포 (평균 1000h)')
plt.xlabel('수명(시간)')
plt.ylabel('확률밀도')
plt.show()

### 정규 분포 — 자연·측정에 널리 (종 모양)

평균 μ를 중심으로 좌우 대칭인 종 모양 분포로, 시험 성적·측정 오차·표본평균에 널리 나타납니다. 평균 μ와 표준편차 σ로 정해집니다.

- 기대값 = μ, scipy `stats.norm(loc=μ, scale=σ)`
- **상위 몇 %의 경계 점수**는 `stats.norm.ppf` 로 거꾸로 구합니다(cdf 의 반대).

<img src="images/정규분포_예시.png" width="780" style="max-width:100%">

In [ ]:
# 정규: 어느 시험 성적 N(70, 10²) — 평균 70점, 표준편차 10점
norm_dist = stats.norm(loc=70, scale=10)
print('P(80점 이상) {:.4f}, P(60점 이하) {:.4f}'.format(norm_dist.sf(80), norm_dist.cdf(60)))
print('상위 10% 커트라인 = {:.1f}점  (이 점수 이상이면 상위 10%)'.format(norm_dist.ppf(0.90)))
print('상위  5% 커트라인 = {:.1f}점'.format(norm_dist.ppf(0.95)))

grid = np.linspace(35, 105, 200)
plt.figure(figsize=(10, 4))
plt.plot(grid, norm_dist.pdf(grid))
plt.axvline(norm_dist.ppf(0.90), color='red', linestyle='--', label='상위 10% 커트라인')
plt.title('정규분포 PDF — 시험 성적 N(70, 10²)')
plt.xlabel('점수')
plt.ylabel('확률밀도')
plt.legend()
plt.show()

### 🖐️ 함께 따라하기 — 설비 알람을 포아송으로 설명할 수 있을까

데모에서는 λ를 임의로 정해 콜센터 전화를 다뤘죠. 이번엔 **데이터에서 λ를 직접 구해** 포아송 분포를 만들고, **이론이 실제와 맞는지** 대조해 봅니다.

설비 알람처럼 '드물게, 서로 독립적으로, 일정한 비율로' 일어나는 사건은 포아송 분포로 설명되는 일이 많습니다. 정말 그런지 확인해 보죠.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) lam = fac['설비알람수'].mean() 으로 관측된 평균 알람 횟수를 구해 출력한다
# 2) dist = stats.poisson(mu=lam) 로 포아송 분포를 만든다
# 3) 이론 확률과 실제 비율을 나란히 출력해 비교한다
#    - P(0번)     : dist.pmf(0)  vs  (fac['설비알람수'] == 0).mean()
#    - P(2번 이하): dist.cdf(2)  vs  (fac['설비알람수'] <= 2).mean()
#    - P(5번 이상): dist.sf(4)   vs  (fac['설비알람수'] >= 5).mean()
# 4) 포아송은 평균과 분산이 같다는 성질이 있습니다 — 실제 평균과 분산(var(ddof=1))도 출력해 비교한다

### ✅ 바로 확인 퀴즈

**1.** "성공확률 p인 시행을 n번 반복했을 때 성공 횟수"는 어떤 분포를 따르나요?

<details><summary>정답 보기</summary>

**이항분포** B(n, p)입니다. 기대값은 np, 분산은 np(1−p)이고 `stats.binom(n, p)` 로 만듭니다.

</details>

**2.** scipy에서 "평균 수명 1000시간"인 지수분포는 `stats.expon(____)` 어떻게 만드나요?

<details><summary>정답 보기</summary>

`stats.expon(scale=1000)` 입니다. scipy의 `scale` 은 **평균 대기시간(1/λ)** 을 뜻하므로 평균 1000시간이면 scale=1000 입니다.

</details>

---
# 7. 정규분포 심화 — 표준화와 68-95-99.7

## 왜 필요할까요?
정규분포는 자연·측정·평균값에서 가장 널리 나타납니다. 평균 μ, 표준편차 σ만 알면 **어떤 값이 나올 확률**을 계산할 수 있고, 단위가 다른 값들도 **표준화(Z-score)** 로 같은 잣대에 놓고 비교할 수 있습니다.

- **Z-score**: Z = (x − μ) / σ. "평균에서 표준편차 몇 칸 떨어졌나". 단위가 사라져 서로 다른 시험 점수도 비교 가능.
- **표준정규분포**: 평균 0, 표준편차 1인 정규분포 `stats.norm(0, 1)`. 모든 정규분포는 표준화하면 여기로 옵니다.

<img src="images/Z_score_개념.png" width="820" style="max-width:100%">

<img src="images/68_95_99.7_법칙.png" width="820" style="max-width:100%">

### 68-95-99.7 법칙(경험규칙)
정규분포에서 평균을 중심으로:
- ±1σ 안에 약 **68%**, ±2σ 안에 약 **95%**, ±3σ 안에 약 **99.7%** 의 값이 들어옵니다.
- 그래서 |Z| > 3 인 값은 1000번에 3번꼴로만 나오는 아주 드문 값 — 이상치 판별의 근거가 됩니다.
- 특정 분위수(예: 상위 5% 경계)는 `stats.norm.ppf(0.95, μ, σ)` 로 거꾸로 구합니다.

In [ ]:
# 표준정규분포로 68-95-99.7 법칙 확인
standard_normal = stats.norm(0, 1)
print('±1σ 안 확률 =', round(standard_normal.cdf(1) - standard_normal.cdf(-1), 4), '  (약 0.68)')
print('±2σ 안 확률 =', round(standard_normal.cdf(2) - standard_normal.cdf(-2), 4), '  (약 0.95)')
print('±3σ 안 확률 =', round(standard_normal.cdf(3) - standard_normal.cdf(-3), 4), '  (약 0.997)')

# 연비 데이터를 Z-score로 표준화
mpg_mean, mpg_std = mpg['mpg'].mean(), mpg['mpg'].std(ddof=1)
print('\nscipy zscore 앞 3개:', np.round(stats.zscore(mpg['mpg'])[:3], 3))
print('연비 40인 차의 Z-score =', round((40 - mpg_mean) / mpg_std, 3), '  (평균보다 표준편차 2칸 위)')
print('상위 5% 연비 경계값   =', round(stats.norm.ppf(0.95, mpg_mean, mpg_std), 2))

## 이 데이터에 정규분포를 써도 될까 — 정규성 확인

여기까지의 계산은 전부 **"이 데이터가 정규분포를 따른다"는 가정** 위에 서 있습니다. 가정이 깨지면 68-95-99.7 법칙도, `norm.cdf` 로 구한 확률도 함께 어긋나요. 그래서 정규분포 공식을 쓰기 **전에** 그 가정이 말이 되는지 확인하는 습관이 필요합니다.

세 가지를 **함께** 봅니다 — 하나만 보고 단정하지 않습니다.

**① 히스토그램에 정규곡선 겹쳐 보기** — 종 모양에서 얼마나 벗어나는지 눈으로 봅니다.

**② Q-Q Plot (`stats.probplot`)** — 데이터의 분위수를 정규분포의 분위수와 짝지어 찍은 그림입니다. **점이 직선 위에 놓이면 정규에 가깝고**, 양 끝이 직선에서 휘면 그쪽 꼬리가 두껍거나 분포가 치우친 것입니다. (오른쪽 끝이 **위로** 휘면 오른쪽 꼬리가 긴 것 = 왜도 > 0)

**③ 왜도·첨도와 68-95-99.7 실측** — §3 에서 배운 왜도·첨도가 **둘 다 0 근처**면 정규에 가깝습니다. 여기에 ±1σ·±2σ 안에 실제로 몇 %가 들어오는지 **직접 세어** 68%·95%와 견주면 가장 직접적인 확인이 됩니다.

> **왜 이게 실무에서 중요한가**: 정규분포가 아닌 데이터에 정규분포 공식을 쓰면 "1000번에 3번" 이라던 사건이 실제로는 훨씬 자주 일어납니다. 품질·금융에서 사고가 나는 전형적인 경로예요.

> 여기까지는 **눈과 요약값으로 보는 판단**입니다. "정규분포가 맞다/아니다"를 **확률로 판정하는 정규성 검정**(Shapiro-Wilk 등)은 다음 단원에서 다룹니다.

In [ ]:
# 정규성 확인 3종 세트 — 연비(mpg)와 가속(acceleration) 을 견줘 본다
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for j, col in enumerate(['mpg', 'acceleration']):
    s = mpg[col].dropna()
    m, sd = s.mean(), s.std(ddof=1)

    # ① 히스토그램 + 정규곡선 — 종 모양과 얼마나 닮았나
    ax = axes[0, j]
    sns.histplot(s, bins=25, stat='density', color='#7fb3d5', ax=ax)
    xs = np.linspace(s.min(), s.max(), 200)
    ax.plot(xs, stats.norm.pdf(xs, m, sd), color='crimson', linewidth=2, label='정규곡선')
    ax.set_title(col + ' — 히스토그램 vs 정규곡선')
    ax.legend()

    # ② Q-Q Plot — 점이 직선에 붙을수록 정규에 가깝다
    ax = axes[1, j]
    (_, _), (_, _, r) = stats.probplot(s, dist='norm', plot=ax)
    ax.set_title(col + ' — Q-Q Plot (직선성 R\u00b2=%.4f)' % r**2)
plt.tight_layout()
plt.show()

# ③ 왜도·첨도 + 68-95-99.7 실측 대조
for col in ['mpg', 'acceleration']:
    s = mpg[col].dropna()
    m, sd = s.mean(), s.std(ddof=1)
    within1 = (abs(s - m) <= sd).mean()
    within2 = (abs(s - m) <= 2 * sd).mean()
    fmt = '%-13s 왜도=%+.3f  첨도=%+.3f  ±1σ 실제 %.1f%% (이론 68.3%%)  ±2σ 실제 %.1f%% (이론 95.4%%)'
    print(fmt % (col, stats.skew(s), stats.kurtosis(s), within1 * 100, within2 * 100))

print('\n가속(acceleration)은 세 지표가 모두 정규에 가깝습니다 — Q-Q 점들이 거의 직선(R²≈0.99).')
print('연비(mpg)는 왜도가 +0.46 으로 오른쪽으로 치우쳐, Q-Q 곡선이 전체적으로 위로 볼록하게 휩니다.')
print('특히 왼쪽 끝이 직선보다 뚜렷하게 위에 있죠 — 정규분포라면 있어야 할 \'아주 낮은 연비\' 차가')
print('실제로는 없다는 뜻입니다. 그래서 ±1σ 안에 63.1% 만 들어옵니다(이론 68.3%).')

### 🖐️ 함께 따라하기 — 당도 규격을 벗어날 확률

데모는 시험 성적으로 정규분포를 다뤘죠. 이번엔 **공장의 품질 규격**에 적용합니다.

이 음료의 당도 규격은 **10.5 ~ 11.5 brix** 입니다. **먼저 당도(`당도_brix`)가 정규분포에 가까운지 확인**한 다음, **규격을 벗어날 확률**과 **상위 5% 커트라인**을 구해 봅시다. 비교를 위해 같은 표의 **내용량(`내용량_ml`)** 도 함께 확인해 봅니다.

> `당도_brix` 에는 당도계 고장으로 측정이 빠진 배치가 있습니다. `dropna()` 로 먼저 제거하세요.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) brix = fac['당도_brix'].dropna() 로 결측을 뺀 당도를 만든다
# 2) 정규성부터 확인한다 — brix 와 fac['내용량_ml'] 각각에 대해
#    왜도(stats.skew)·첨도(stats.kurtosis)와 ±1σ 안에 실제로 든 비율을 출력해 견준다
# 3) 두 변수의 Q-Q Plot 을 나란히 그린다 (stats.probplot(s, dist='norm', plot=ax))
#    → 어느 쪽에 정규분포 공식을 써도 되는지 판단한다
# 4) 평균(mu)과 표준편차(sd, ddof=1)를 구해 소수 셋째 자리로 출력한다
# 5) nd = stats.norm(loc=mu, scale=sd) 로 정규분포를 만든다
# 6) 규격 미달 P(10.5 미만)은 nd.cdf(10.5), 초과 P(11.5 초과)는 nd.sf(11.5) 로 구해 출력한다
# 7) 둘을 더한 '규격 이탈률'을 출력하고, 실제 데이터의 이탈 비율과 비교한다
# 8) nd.ppf(0.95) 로 상위 5% 커트라인 당도를 출력한다

### ✅ 바로 확인 퀴즈

**1.** 정규분포에서 평균 ±2σ 구간에는 전체의 약 몇 %가 들어오나요?

<details><summary>정답 보기</summary>

약 **95%** 입니다(정확히는 95.45%). ±1σ는 약 68%, ±3σ는 약 99.7% 입니다.

</details>

**2.** 국어 Z-score가 +1.5, 수학 Z-score가 +0.8 인 학생은 두 과목 중 어느 쪽을 상대적으로 더 잘한 것인가요?

<details><summary>정답 보기</summary>

**국어**입니다. Z-score가 클수록 평균에서 위로 더 멀리 있다는 뜻이라, 원점수 단위와 무관하게 국어에서 상위권에 더 가깝습니다. 이처럼 Z-score는 단위·평균·표준편차가 다른 값들을 같은 잣대로 비교하게 해 줍니다.

</details>

**3.** 어떤 데이터의 Q-Q Plot 에서 점들이 **오른쪽 끝만 직선 위로 크게 휘어** 올라갔습니다. 이 분포의 왜도는 0보다 클까요, 작을까요? 그리고 이 데이터에 `norm.cdf` 로 확률을 구해도 될까요?

<details><summary>정답 보기</summary>

왜도는 **0보다 큽니다**(오른쪽 꼬리가 길다). Q-Q 오른쪽 끝이 위로 휘었다는 건 큰 값 쪽이 정규분포가 예상한 것보다 **더 크게 뻗어 있다**는 뜻이거든요.

이런 데이터에 `norm.cdf` 를 그대로 쓰면 **오른쪽 꼬리의 확률을 실제보다 작게** 잡습니다 — "좀처럼 안 일어난다"고 계산한 일이 현실에서는 꽤 자주 일어나는 것이죠. 휜 정도가 크면 정규 가정을 쓰지 말거나, 로그 변환처럼 분포를 펴 주는 처리를 먼저 고려합니다.

</details>

---
## 🚀 응용 클론코딩 — 품질 리포트 한 장 만들기

오늘 배운 것을 **한 흐름**으로 이어 봅시다: 대표값·산포 → 이상치 → 상관 → 확률.

**미션**: 공장 품질 데이터로 **라인별 품질 요약 리포트**를 만들고, "어느 라인이 문제인가"와 "규격을 벗어날 확률은 얼마인가"에 답합니다.

> 숫자 하나하나는 이미 다 배웠습니다. 이제 그것들을 **묶어서 하나의 결론**으로 만드는 연습입니다.

In [ ]:
# 🖐️ 함께 따라하기 — 품질 리포트 (아래 순서대로 직접 작성해 보세요)
# 1) 라인(A·B·C)별로 내용량_ml 의 평균·중앙값·표준편차를 구해 출력한다
#    (힌트: fac.groupby('라인')['내용량_ml'].agg([...]) — 6일차에 배운 groupby)
# 2) 라인별 불량수 평균도 구해, 어느 라인이 가장 나쁜지 확인한다
# 3) 불량수와 라인속도_bpm, 불량수와 숙련도_년 의 피어슨 상관을 각각 구해 출력한다
# 4) 전체 내용량의 평균·표준편차로 정규분포를 만들어, 규격(495~505ml)을 벗어날 확률을 구한다
# 5) 위 결과를 근거로 '무엇이 문제이고 무엇을 조치할지' 한 문단으로 정리한다

### ✅ 바로 확인 퀴즈

**1.** 위 리포트에서 '라인속도를 낮추면 불량이 줄어든다'고 **단정**할 수 있나요?

<details><summary>정답 보기</summary>

**단정할 수 없습니다.** 상관은 인과가 아닙니다. 속도가 빠른 라인은 신형이라 다른 조건(교대조 구성·설비 특성)도 함께 다를 수 있어요. 실제로 확인하려면 **속도만 바꿔 보는 실험**이 필요합니다.

</details>

**2.** 정규분포로 계산한 이탈률이 실제 이탈률과 다르다면, 무엇을 의심해야 할까요?

<details><summary>정답 보기</summary>

**데이터가 정규분포를 따르지 않을 가능성**입니다. 충전량에는 밸브 고장으로 생긴 극단적인 이상치가 섞여 있어 좌우대칭이 아니거든요. 정규분포 공식은 '정규분포를 따른다'는 가정 위에서만 정확합니다.

</details>

---
## 이번 강의 정리

| 주제 | 핵심 도구 | 한 줄 요약 |
|---|---|---|
| 대표값 | `mean` `median` `mode` `trim_mean` `np.average` | 분포의 중심. 이상치엔 중앙값이 강건 |
| 산포도 | `var/std(ddof=1)` `stats.iqr` `CV` | 퍼짐. n−1 베셀 보정, 단위 다르면 CV |
| 분포 형태 | `stats.skew` `stats.kurtosis` | 왜도=치우침, 첨도=꼬리 두께(꼬리 리스크) |
| 상관 분석 | `pearsonr` `spearmanr` `corr` `heatmap` | 관계의 방향·세기. 상관 ≠ 인과 |
| 확률·기대값 | 덧셈·곱셈·조건부, `E[X]` `Var(X)` | 불확실성의 언어. 대수의 법칙 |
| 확률분포 | `bernoulli` `binom` `poisson` `uniform` `expon` `norm` | PMF/PDF/CDF, `.cdf/.sf/.ppf` |
| 정규분포 | Z-score, 68-95-99.7, `ppf` | 표준화로 비교, 경험규칙 |
| 정규성 확인 | 히스토그램+정규곡선, `stats.probplot`(Q-Q), 왜도·첨도 | 정규분포 공식을 **쓰기 전에** 가정부터 확인 |

이제 데이터를 **숫자로 요약**하고, 두 변수의 관계를 재고, 확률분포로 불확실성을 다룰 수 있습니다.

## ⏭️ 예고 — 다음 시간: 통계적 추론

이번 시간엔 **가진 데이터를 요약**했습니다. 다음 시간엔 그 데이터(표본)로 **보지 못한 전체(모집단)를 추측**합니다.
- **중심극한정리(CLT)** — 표본평균은 왜 정규분포를 따르는가
- **신뢰구간** — 표본으로 **모평균이 있을 만한 범위를 추정**하기
- **신뢰구간으로 주장 판단** — 누군가의 주장이 데이터와 맞는지 가늠하기

오늘 익힌 평균·표준편차·정규분포·확률이 그 모든 추론의 **재료**가 됩니다. 수고하셨습니다!